<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Operating-Systems/06-address-spaces-page-tables-and-tlb.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Operating Systems guideline](Operating-Systems.html)


## **Address Spaces, Page Tables, and the TLB**

A machine instruction such as `load [address]` appears to name memory directly, but a modern process normally does not issue a raw DRAM address. The CPU first forms an **effective address** according to the instruction set. In user mode that address belongs to the process's **virtual address space**. The Memory Management Unit (MMU) then translates it, checks permissions, and only afterward sends a physical address toward the cache and memory hierarchy.

This is a different question from the memory ordering discussed in the synchronization chapter. A language or hardware memory model asks **when one CPU may observe another CPU's writes**. Virtual memory asks **which physical storage a virtual address denotes and whether this access is allowed**. Both participate in one memory operation, but they solve different correctness problems.

The chapter follows one reference from virtual address to physical byte. It first builds the address-space abstraction, develops relocation and paging, then adds page-table structures, the TLB fast path, exceptions, and architecture-specific details. Demand loading, replacement, swap, and physical allocation are deliberately deferred to the next chapter.

### **Why Virtualize Memory?**

If applications used physical addresses directly, every program would need to know where free RAM happened to exist. Loading one program could force another to move; a stray pointer could overwrite the kernel; two programs could not conveniently use the same link address; and reserving a large but mostly unused range would immediately consume scarce memory.

An **address space** is the set of virtual addresses a process is permitted to use, together with mappings and policies that explain those addresses. It works like a building's room-number plan: programs refer to stable room numbers, while a controlled directory maps each number to an actual room. The directory can deny a number, map two tenants to different rooms with the same local number, or deliberately let them share one room.

The operating system owns the mapping policy, while processor hardware enforces the active translation on every instruction fetch, load, and store. This hardware-software contract aims for three properties:

- **transparency**: ordinary code uses addresses without performing translations itself;
- **efficiency**: the common case completes near cache-access speed;
- **protection**: a process cannot bypass mapping and permission rules merely by constructing a pointer.

![Virtual address-space properties mapped onto shared physical memory.](assets/address-space-virtualization.svg){fig-alt="A process virtual layout maps private pages to different physical frames, may share a selected frame with another process, and leaves unmapped holes without data frames." width="98%"}

*Figure: original explanatory diagram based on the [Linux virtual-memory concepts](https://docs.kernel.org/admin-guide/mm/concepts.html) and the [OSTEP address-space chapter](https://pages.cs.wisc.edu/~remzi/OSTEP/vm-intro.pdf).*

Virtual memory is sometimes described only as "using disk as extra RAM." That is too narrow. Swapping and demand paging are optional policies built on top of address translation. Relocation, isolation, controlled sharing, sparse mappings, memory-mapped files, copy-on-write, and executable permissions remain valuable even when every active page is already resident in RAM.

#### **Relocation, Isolation, Sharing, and Sparsity**

The abstraction earns its complexity because one mapping mechanism supports several goals at once.

| Goal | Mapping idea | Concrete example | Limitation to remember |
|---|---|---|---|
| Relocation | The same virtual page may point to any suitable frame | Two instances of one executable can use the same link-time addresses | Pointers are stable only within the intended address space and lifetime |
| Isolation | Unmapped and supervisor-only pages reject unauthorized access | Process A cannot name Process B's private heap through its own page table | Hardware isolation still depends on correct kernel page-table management |
| Sharing | Distinct virtual mappings may point to the same frame | Shared-library text, shared memory, and file-backed pages | Writers need synchronization and carefully chosen sharing semantics |
| Sparsity | Missing mappings represent unused ranges cheaply | A large stack limit or arena can reserve address range without backing every page | Virtual capacity is not a promise that physical memory will later be available |

Relocation allows Address Space Layout Randomization (ASLR), position-independent executables, dynamically loaded libraries, and flexible placement of physical frames. Isolation comes from both **absence** and **permission**: no valid page-table entry means there is no translation, while a present entry can separately disallow writes, execution, or user-mode access.

Sharing does not violate private address spaces. The privacy applies to each process's mapping namespace; the kernel may intentionally install entries in two namespaces that point to one physical frame. Conversely, two processes can contain the identical numeric pointer `0x400000` while reaching unrelated frames.

Sparsity is especially important in 64-bit systems. A process can place the heap, mappings, stacks, and guard regions far apart. Only populated portions require leaf mappings and, with multilevel tables, only populated branches require lower-level table pages.

### **Address Spaces and Memory Regions**

An address space is not just a bag of page-table entries. The kernel also maintains higher-level **regions**: intervals with a common purpose, permission policy, sharing mode, and backing object. Linux commonly calls these Virtual Memory Areas (VMAs). One VMA might describe a read-only executable segment backed by an ELF file; another might describe a private anonymous stack.

This creates two related but distinct questions:

1. **Region question**: does the process logically own this interval, and is this operation allowed in principle?
2. **translation question**: does a usable PTE currently map this particular page, and where does it point?

A valid region may contain pages that have not yet been materialized. That distinction is what later permits lazy allocation and demand paging. It also explains why inspecting `/proc/PID/maps` shows ranges and permissions, not a reliable list of resident physical frames.

![A conceptual Linux user address space and its region metadata.](assets/process-address-space-regions.svg){fig-alt="Conceptual process address space showing executable text, data, heap, unmapped gaps, mapped files and libraries, guard page, and stack, alongside VMA and page-table metadata." width="84%"}

*Figure: original explanatory diagram based on Linux [`/proc/PID/maps`](https://man7.org/linux/man-pages/man5/proc_pid_maps.5.html) semantics. Exact placement varies by ABI, loader, ASLR, allocator, and process history.*

On Linux, the following commands provide complementary views:

```bash
# Region view for the current shell process.
head -n 15 /proc/$$/maps

# Loadable ELF segments requested by the executable file.
readelf -W -l /proc/$$/exe

# Base-page size exposed to this user-space environment.
getconf PAGE_SIZE
```

`readelf` describes what the loader finds in the executable; `/proc/$$/maps` shows the process after the loader, dynamic linker, allocator, and libraries have added mappings. Neither command alone is the complete story.

#### **Code, Data, Heap, Stack, and Mapped Regions**

The traditional labels are useful, but they should not be mistaken for one fixed contiguous layout:

- **text** contains executable instructions and is normally readable and executable but not writable;
- **read-only data** contains constants and relocation results that should not change;
- **initialized data** starts with bytes stored in the executable;
- **BSS** describes zero-initialized objects without storing all those zeros in the file;
- **heap** is a logical source of dynamic storage, often implemented through both `brk` growth and anonymous `mmap` arenas;
- **stack** stores call state and automatic objects, usually with an inaccessible guard region to turn runaway growth into a fault;
- **mapped regions** include shared libraries, memory-mapped files, anonymous mappings, shared memory, JIT-managed memory, and additional thread stacks.

The ELF loader maps **segments**, not source-language sections one by one. Multiple sections with compatible permissions can occupy one loadable segment. Likewise, `malloc()` does not necessarily extend one simple heap boundary: allocators may request separate anonymous mappings for large blocks or parallel arenas.

Permissions should follow the intended use. Writable data usually should not be executable, executable code usually should not be writable, and guard pages should be neither. This **W^X** discipline reduces the opportunity to turn a memory-corruption bug into execution of injected bytes. JIT runtimes commonly write code in one phase and then change the mapping to executable in another, rather than leaving it writable and executable indefinitely.

### **Early Address-Translation Mechanisms**

Paging is easier to understand after seeing the problem solved with less flexible hardware. Early mechanisms also expose the two responsibilities that remain today: **relocate** a program-generated address and **reject** an invalid access.

#### **Base and Bounds**

In dynamic relocation, the process sees a virtual interval beginning at zero. Two privileged registers describe its physical placement:

- `base` is the physical start of the process allocation;
- `bounds` or `limit` is the size of the legal virtual interval.

For a byte address `VA`, the MMU performs:

$$
PA = base + VA \quad \text{only if} \quad 0 \le VA < bounds
$$

`VA` is the program-generated virtual address, `PA` is the resulting physical address, and `bounds` prevents the addition from reaching another allocation. For `base = 0x4000`, `bounds = 0x800`, and `VA = 0x320`, the access is valid and produces `PA = 0x4320`. `VA = 0x900` raises a protection exception instead.

```text
translate(VA, operation):
    if VA >= bounds:
        raise protection_fault
    PA = base + VA
    perform operation at PA
```

![Base-and-bounds relocation compared with segmentation.](assets/base-bounds-segmentation.svg){fig-alt="Base and bounds relocate one contiguous process allocation; segmentation relocates code, heap, and stack independently but leaves external holes in physical memory." width="98%"}

*Figure: original explanatory diagram based on the [OSTEP address-translation](https://pages.cs.wisc.edu/~remzi/OSTEP/vm-mechanism.pdf) and [segmentation](https://pages.cs.wisc.edu/~remzi/OSTEP/vm-segmentation.pdf) chapters.*

Base and bounds are fast: one comparison and one addition are conceptually enough. A context switch saves and restores the registers, giving the next process a different view. The central weakness is contiguity. The process's code, unused gap, heap, and reserved stack must fit in one physical interval. Growth may require moving the whole allocation or finding a larger hole.

#### **Segmentation and External Fragmentation**

Segmentation gives independently meaningful regions separate translation records. A virtual address contains or implies a segment selector plus an offset. Each segment descriptor carries a base, limit, permissions, and sometimes a growth direction:

$$
PA = base[s] + offset \quad \text{if the offset and operation satisfy segment } s
$$

Code can be read/execute, data can be read/write, and a stack can grow independently. Segments may also be shared. This matches how programmers think about regions more closely than one base-and-bounds pair.

The cost is **external fragmentation**. Variable-size segments leave physical holes between allocations. Even if the holes contain enough total bytes, no individual hole may fit a new segment. Compaction can create a larger hole but requires copying live data and updating placement, which is expensive and difficult while programs run.

External fragmentation is free space split **between** allocations. Internal fragmentation, introduced below, is unused space **inside** an allocated unit. Paging largely replaces external fragmentation with smaller internal waste by making every placement unit the same size.

Modern x86-64 software still has architectural segmentation remnants, notably `FS` and `GS` bases used for thread-local or per-CPU data, but ordinary user virtual memory primarily relies on paging. The historical progression matters because paging did not remove protection and relocation; it changed their allocation granularity.

### **Paging**

Paging divides virtual memory into fixed-size **pages** and physical memory into same-size **frames**. A page can occupy any suitable frame, so logical neighbors do not need to be physical neighbors. A page table records the page-to-frame relationship. The address-space overview above already shows the essential result: consecutive virtual pages may map to unrelated frames, while the translation preserves each byte's offset inside its page.

Fixed-size frames solve the placement problem: any free frame can hold any base page. They also make allocation and reclamation easier to describe. The price is a potentially enormous translation structure and an extra lookup before ordinary memory access. Page-table hierarchies reduce the space cost; the TLB reduces the common-case time cost.

#### **Pages, Frames, and Offset Preservation**

Let page size be $P = 2^p$ bytes. The low $p$ bits select a byte within the page; all higher virtual bits form the Virtual Page Number (VPN):

$$
VPN = \left\lfloor \frac{VA}{P} \right\rfloor, \qquad offset = VA \bmod P
$$

The page table maps `VPN` to a Physical Frame Number (PFN), also called a Physical Page Number (PPN). Translation then reconstructs:

$$
PA = PFN \times P + offset
$$

The offset is unchanged because pages and frames have equal size and alignment. Only the identity of the containing page changes.

![Step-by-step paging translation that preserves the page offset.](assets/paging-offset-translation-animated.svg){fig-alt="Animated worked example splitting virtual address 0x2D3A into VPN 0x0B and offset 0x13A, mapping to PFN 0x15, and producing physical address 0x553A." width="98%"}

*Figure: original animated explanatory diagram. The calculation uses a 1 KiB page only to keep the hexadecimal example compact.*

In the example, $P = 1024 = 0x400$ bytes. `VA 0x2D3A` therefore has `VPN 0x0B` and offset `0x13A`. Mapping VPN `0x0B` to PFN `0x15` gives `PA 0x553A`. If the PTE denies the requested operation, the arithmetic result is irrelevant: permission checking is part of translation.

This fixed split also explains alignment. A page-aligned virtual address has all offset bits zero. Page-table entries point to page-aligned frames, so their low address bits are available to encode flags rather than a byte position.

#### **Internal Fragmentation and Page Size**

Paging can waste the unused tail of an allocated last page. If object or region endings are uniformly distributed within a page, the expected tail waste for one independently rounded allocation is approximately $P/2$. This is a modeling rule, not a claim that every object wastes half a page: allocators pack many small objects into the same page, and file-backed regions may share page-cache storage.

Page size affects several competing costs:

- smaller pages provide fine-grained protection, sharing, copying, and sparse allocation;
- larger pages need fewer PTEs and increase the amount of memory reachable through a fixed number of TLB entries;
- larger pages can increase unused tail space and require larger contiguous physical extents;
- moving, clearing, copying, or evicting a large page transfers more data;
- hardware often supports several sizes, allowing a workload-specific mixture.

The **TLB reach** of $N$ equal-sized entries is approximately:

$$
reach = N \times P
$$

With 64 entries, 4 KiB pages cover 256 KiB, while 2 MiB pages cover 128 MiB. Associativity, separate instruction/data TLBs, multiple TLB levels, and mixed page sizes make real hardware more complicated, but the basic pressure remains.

![Page-size tradeoffs for a 64 MiB mapping.](assets/page-size-tradeoffs.svg){fig-alt="Comparison of 4 KiB and 2 MiB pages showing mapping count, TLB reach, metadata cost, allocation granularity, and fragmentation tradeoffs." width="98%"}

*Figure: original explanatory diagram. Counts assume a dense 64 MiB mapping and compare equal 64-entry translation capacities for illustration.*

Huge pages are most useful for large, dense, stable working sets. They are less attractive for sparse ranges, frequently changed permissions, small mappings, or systems unable to obtain suitably aligned contiguous frames. The next chapter discusses transparent huge pages and physical-allocation consequences in more detail.

### **Page-Table Entries and Permissions**

A Page-Table Entry (PTE) is not merely `VPN -> PFN`. It combines an address with the state needed to validate and control access. Exact bit names and encodings vary, but common concepts include:

- **valid or present**: whether the entry currently supplies a usable translation;
- **read, write, and execute** permissions;
- **user or supervisor** accessibility;
- **accessed/reference** state, indicating recent use;
- **dirty/modified** state, indicating a write since the relevant reset;
- **global** or address-space-tag behavior for selected mappings;
- **cacheability and memory-type** controls;
- **software-owned bits** used by the OS for states such as copy-on-write or swapped entries.

![Generic PTE fields and the resulting access decision.](assets/pte-permissions-and-faults.svg){fig-alt="Generic page-table entry with frame number, valid, read, write, execute, user, accessed, dirty, and software bits, followed by validity and permission checks." width="98%"}

*Figure: original explanatory diagram based on the [Linux page-table overview](https://docs.kernel.org/mm/page_tables.html), Intel architecture manuals, and the RISC-V supervisor specification.*

One subtlety is that **valid**, **present**, and **resident** are not universal synonyms. An architecture's valid bit says whether hardware accepts the entry in its defined form. An operating system may encode additional nonpresent states in software. Region metadata may say an address is legal even when the current leaf PTE deliberately causes a fault so the kernel can allocate, load, or copy a page.

#### **Presence, Access, Dirty, Execute, and User Bits**

For each memory operation, the MMU conceptually combines several facts:

1. Is the virtual address architecturally well formed?
2. Does the page-table walk reach a valid leaf?
3. Does the leaf permit this read, write, or instruction fetch?
4. Does current privilege permit access to this user/supervisor page?
5. Are accessed/dirty semantics already satisfied, or must hardware or software update them?

Execute permission is separate from read permission on many modern architectures. A page can hold readable non-executable data, or executable code that is not writable. Dirty state helps the OS decide whether a file-backed or swapped page must be written before reuse. Accessed state provides imperfect evidence for replacement algorithms because hardware behavior and clearing protocols are architecture-specific.

POSIX `mprotect()` exposes permission changes at region granularity to user programs; the kernel implements them by updating relevant mappings and making cached translations coherent. The following program changes one anonymous page from read/write to read-only and back without deliberately triggering a fatal fault.

<details>
<summary><strong>C: changing page permissions with mmap and mprotect</strong></summary>

```c
#define _GNU_SOURCE
#include <errno.h>
#include <stdio.h>
#include <string.h>
#include <sys/mman.h>
#include <unistd.h>

int main(void) {
    long page_size = sysconf(_SC_PAGESIZE);
    if (page_size <= 0) {
        fputs("cannot determine page size\n", stderr);
        return 1;
    }

    char *page = mmap(NULL, (size_t)page_size,
                      PROT_READ | PROT_WRITE,
                      MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (page == MAP_FAILED) {
        perror("mmap");
        return 1;
    }

    // The mapping initially permits ordinary stores.
    snprintf(page, (size_t)page_size, "translation includes permissions");

    // mprotect operates on page-aligned mappings and changes future access.
    if (mprotect(page, (size_t)page_size, PROT_READ) == -1) {
        perror("mprotect read-only");
        munmap(page, (size_t)page_size);
        return 1;
    }

    printf("read-only page says: %s\n", page);
    // Writing page[0] here would violate the mapping and normally raise SIGSEGV.

    if (mprotect(page, (size_t)page_size, PROT_READ | PROT_WRITE) == -1) {
        perror("mprotect restore");
        munmap(page, (size_t)page_size);
        return 1;
    }

    page[0] = 'T';
    if (munmap(page, (size_t)page_size) == -1) {
        perror("munmap");
        return 1;
    }
    return 0;
}
```

```bash
cc -std=c11 -O2 -Wall -Wextra -Wpedantic permissions.c -o permissions
./permissions
```

</details>

The API contract comes from POSIX [`mmap()`](https://pubs.opengroup.org/onlinepubs/9799919799/functions/mmap.html), [`mprotect()`](https://pubs.opengroup.org/onlinepubs/9799919799/functions/mprotect.html), and [`munmap()`](https://pubs.opengroup.org/onlinepubs/9799919799/functions/munmap.html). A production program must also coordinate permission changes with other threads; changing a mapping while another thread executes or writes it is a synchronization and design problem, not just a system-call problem.

### **Page-Table Organizations**

A page table is a data structure indexed by virtual-page information. The most direct design is an array with one PTE per virtual page. That design makes lookup simple but scales with the **maximum virtual address range**, including holes.

For a $V$-bit virtual address, page size $P$, and $E$ bytes per entry, a full linear table requires:

$$
table\ bytes = \frac{2^V}{P} \times E
$$

With 32-bit addresses, 4 KiB pages, and 4-byte PTEs, the table occupies $2^{20} \times 4 = 4$ MiB per process. With 48 translated bits, 4 KiB pages, and 8-byte PTEs, a hypothetical flat table would occupy $2^{36} \times 8 = 512$ GiB per process. Most of it would describe unmapped holes.

#### **Linear and Multilevel Page Tables**

A **multilevel** or **radix-tree** page table splits the VPN into indexes. A top-level entry points to a lower table only when some address in that branch is mapped. With 4 KiB table pages and 8-byte entries, each table contains 512 entries, so each level consumes 9 VPN bits.

![A four-level page-table walk with sparse branches.](assets/multilevel-page-table-walk.svg){fig-alt="Four-level radix page table using four nine-bit virtual-address indexes and a twelve-bit offset, with only the selected sparse branch allocated and a leaf PTE pointing to a data frame." width="98%"}

*Figure: original explanatory diagram based on the [Linux page-table hierarchy](https://docs.kernel.org/mm/page_tables.html) and [OSTEP smaller-page-table chapter](https://pages.cs.wisc.edu/~remzi/OSTEP/vm-smalltables.pdf).*

The translation is a dependent walk:

```text
table = physical_address(root_register)
for index from highest VPN field to lowest:
    entry = read(table[index])
    if entry is invalid:
        raise page_fault
    if entry is a leaf:
        check permissions
        combine leaf PFN with remaining VPN bits and page offset
        return physical_address
    table = physical_address(entry.next_table)
```

Stopping at an upper-level leaf creates a large-page mapping. The unused lower VPN bits become part of the offset within that large page, and both virtual and physical addresses must meet the required alignment.

Linux presents architecture-independent code with a hierarchy named PGD, P4D, PUD, PMD, and PTE. Hardware may implement fewer levels; Linux **folds** unused software levels so generic traversal code remains structurally consistent. The key benefit is proportional allocation: a sparse address space needs table pages only for populated branches. The cost is more dependent memory reads on a TLB miss.

#### **Hashed and Inverted Page Tables**

Radix trees are not the only possible organization. A **hashed page table** hashes `(address-space identifier, VPN)` into a bucket, then compares full keys along a chain or probe sequence. It can represent sparse spaces compactly without reserving a conceptual slot for every VPN.

An **inverted page table** turns the relationship around: it maintains roughly one record per physical frame, recording which address space and VPN currently owns that frame. Its size scales with physical memory rather than the sum of virtual ranges.

![Hashed and inverted page-table organizations.](assets/alternative-page-table-organizations.svg){fig-alt="Hashed page table mapping an ASID and VPN through a bucket chain, compared with an inverted table storing one owner record per physical frame." width="98%"}

*Figure: original explanatory diagram based on the [OSTEP smaller-page-table discussion](https://pages.cs.wisc.edu/~remzi/OSTEP/vm-smalltables.pdf).*

| Organization | Space scales with | Lookup strength | Main complication |
|---|---|---|---|
| Linear | maximum virtual pages per address space | direct array indexing | enormous sparse tables and contiguous table storage |
| Multilevel radix | populated virtual branches | deterministic hardware walk | several dependent reads on a miss |
| Hashed | populated mappings and hash load | good for huge sparse spaces | collisions, chaining, and less regular walking |
| Inverted | physical frames | compact when virtual space greatly exceeds RAM | reverse lookup, aliases, shared pages, and hashing |

Hardware-managed translation generally favors a documented, regular format such as a radix tree. Architectures with software-managed TLB misses can give the OS more freedom because software may consult any suitable structure before inserting a TLB entry.

### **The MMU Address-Translation Path**

The MMU sits on the critical path of every instruction fetch and ordinary data access, so the design separates a fast cached path from a slower page-table walk. The exact microarchitecture is implementation-specific, but the conceptual flow is stable.

![MMU flow through a TLB hit, page-table walk, or page fault.](assets/mmu-tlb-page-walk-animated.svg){fig-alt="Animated MMU flowchart where a tagged TLB hit proceeds to permission checking and a physical address, while a miss invokes a page-table walk that refills the TLB or raises a page fault." width="98%"}

*Figure: original animated explanatory diagram based on the [Linux MMU, TLB, and page-fault overview](https://docs.kernel.org/mm/page_tables.html).*

For one access, the hardware conceptually performs:

1. The instruction's address-generation logic produces a virtual address and access type.
2. The MMU checks a TLB using the VPN, page size, and current address-space tag.
3. On a hit, cached permissions are checked and the PFN is combined with the offset.
4. On a miss, a page-table walker starts from the privileged root register and reads paging structures.
5. A valid permitted leaf may refill the TLB and restart or complete the access.
6. A malformed address, invalid entry, or denied operation raises a synchronous exception.
7. After translation, the physical address participates in the ordinary cache and memory lookup.

A page-table walk itself performs memory accesses. Those accesses use physical addresses derived from the root and non-leaf entries rather than recursively translating ordinary user pointers. Modern processors may cache upper-level entries in **page-walk caches**, overlap parts of translation with cache indexing, maintain separate instruction and data TLBs, and support several page sizes.

Page-table memory is shared state between the kernel and MMU. The kernel must publish changes with architecture-defined ordering and invalidate stale cached translations. A lock that protects the software data structure does not automatically remove an old entry from another CPU's TLB.

The most important diagnostic distinction is:

- **TLB miss**: the cached translation is absent, so hardware or privileged software must find it;
- **page fault**: translation or access validation cannot proceed normally and transfers control to the kernel.

A TLB miss can complete without a page fault, and a protection fault can occur even when the translation is cached.

### **Translation Lookaside Buffers**

A Translation Lookaside Buffer is a cache of recently used translations and their permissions. It is not a cache of application bytes. A typical lookup key includes a virtual-page tag, an address-space identifier, and enough page-size information to interpret the offset. The value includes a PFN and access-control state.

TLBs exploit locality. Sequential instruction fetches and array accesses repeatedly touch addresses within the same few pages, so one cached translation serves many byte accesses. Separate instruction and data TLBs may feed a shared second-level translation cache; actual capacities and associativities vary by microarchitecture.

#### **TLB Hits, Misses, and Replacement**

A TLB **hit** finds a matching tag with usable permissions. A **miss** starts a page-table walk or, on some architectures, traps to privileged software that fills the TLB. Misses can be classified like other cache misses:

- **cold miss**: the translation has not been used since creation, invalidation, or address-space activation;
- **capacity miss**: the active translation working set exceeds available entries;
- **conflict miss**: several hot VPNs compete for the same set in a set-associative TLB;
- **coherence/invalidation miss**: the kernel deliberately removed an entry after a mapping change.

Replacement is commonly hardware-controlled and not architecturally specified. Approximate LRU, pseudo-random, or implementation-specific policies are possible. Software should optimize locality and page size rather than relying on one undocumented replacement rule.

Ignoring overlap and page-walk caches, a useful normalized estimate is:

$$
EAT = h \times 1 + (1-h) \times (L+1)
$$

`EAT` is effective access cost measured in memory-access units, $h$ is TLB hit rate, and $L$ is the number of paging-structure reads on a miss. The final `+1` is the actual data access. With a four-level walk and $h=0.99$, the estimate is $0.99 \times 1 + 0.01 \times 5 = 1.04$. At $h=0.95$, it becomes $1.20$. This simplified model shows why a few percentage points matter; real latency depends heavily on cache hits, parallelism, speculation, and the memory hierarchy.

Linux performance counters can reveal translation pressure, although event availability and names vary by CPU and security policy:

```bash
# Event names are platform-dependent; use `perf list | grep -i tlb` first.
perf stat -e dTLB-loads,dTLB-load-misses,iTLB-load-misses ./program

# System-wide page-table and huge-page accounting offers a different view.
grep -E 'PageTables|HugePages|AnonHugePages' /proc/meminfo
```

The Linux x86 [TLB documentation](https://docs.kernel.org/arch/x86/tlb.html) emphasizes a practical invalidation tradeoff: invalidating individual pages costs repeated instructions, while flushing broadly destroys unrelated useful entries that must later be refilled.

#### **ASIDs, PCIDs, and Context Switches**

Without tags, a TLB entry for VPN `0x40` from Process A is unsafe after switching to Process B, because B may use the same VPN for a different frame. The simplest correction is to flush non-global entries whenever the page-table root changes. That is correct but creates cold misses after every switch.

An **Address Space Identifier (ASID)** tags translations with the address space that created them. x86 calls its related facility a **Process-Context Identifier (PCID)**. A context switch can select another root and tag while retaining entries for both processes. Tags are finite and eventually reused, so the OS needs generation or invalidation rules before assigning an old tag to a new address space.

![Tagged TLB entries and a multicore TLB shootdown.](assets/tagged-tlb-context-switch-shootdown.svg){fig-alt="Context switch retaining ASID or PCID tagged translations for two processes, followed by a kernel mapping update that sends invalidation requests to other CPUs and waits for acknowledgement." width="98%"}

*Figure: original explanatory diagram based on Linux [PCID/PTI](https://docs.kernel.org/arch/x86/pti.html) and [TLB invalidation](https://docs.kernel.org/arch/x86/tlb.html) documentation.*

Tags solve identity, but mapping updates still require coherence. If an address space ran on several CPUs, each CPU may cache the old PTE. After unmapping a page, changing permissions, or reusing the frame, the kernel sends a **TLB shootdown** to CPUs that might hold the stale translation. Those CPUs invalidate the target entry or range and acknowledge as required before reclamation proceeds.

Shootdowns are costly because they combine interprocessor communication, serialization, and later TLB refill work. Kernels batch changes and track which CPUs used an address space to reduce unnecessary messages. However, invalidation cannot be delayed past a point where stale access would violate lifetime or permission guarantees.

### **Page Faults as Controlled Exceptions**

A page fault is a synchronous architectural exception associated with a particular instruction and virtual access. The CPU records enough state for the kernel to identify the address, operation, privilege mode, and broad cause. The kernel then decides whether the access is repairable or erroneous.

![Decision flow for classifying and handling a page fault.](assets/page-fault-classification.svg){fig-alt="Decision tree checking address form, mapped region, requested permission, and present translation before rejecting the access, repairing a valid missing page, refreshing state, and restarting the instruction." width="98%"}

*Figure: original explanatory diagram based on the Linux [MMU and page-fault overview](https://docs.kernel.org/mm/page_tables.html). Detailed demand-paging actions are developed in chapter 07.*

Common classes include:

- an address with no legal region, such as a wild pointer;
- a protection violation, such as writing a read-only mapping or executing an NX page;
- a legal anonymous page that needs zero-filled storage;
- a private write that requires copy-on-write;
- a legal file-backed page not currently present;
- accessed/dirty state requiring architecture-specific software assistance;
- a stale or incomplete translation state that can be refreshed.

Repairable faults update metadata and PTEs, invalidate or refill translation state as required, and restart the faulting instruction. The instruction must behave as if it had not partially committed an architecturally visible memory operation. Unrepairable user accesses normally become signals such as `SIGSEGV` or `SIGBUS`; kernel faults require carefully designed recovery paths or indicate a kernel bug.

The following Linux program maps anonymous virtual space and then writes one byte per base page. `mmap()` creates the region, while first touch commonly causes minor faults that allocate zero-filled pages. Exact counts vary because the kernel may use huge pages, pre-faulting, accounting optimizations, or other policies.

<details>
<summary><strong>C: observing first-touch minor faults on Linux</strong></summary>

```c
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <sys/mman.h>
#include <sys/resource.h>
#include <unistd.h>

static long minor_faults(void) {
    struct rusage usage;
    if (getrusage(RUSAGE_SELF, &usage) == -1) {
        perror("getrusage");
        exit(EXIT_FAILURE);
    }
    return usage.ru_minflt;
}

int main(void) {
    const size_t length = 64U * 1024U * 1024U;
    long page_size = sysconf(_SC_PAGESIZE);
    if (page_size <= 0) {
        fputs("cannot determine page size\n", stderr);
        return 1;
    }

    long before_map = minor_faults();
    volatile unsigned char *memory = mmap(
        NULL, length, PROT_READ | PROT_WRITE,
        MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (memory == MAP_FAILED) {
        perror("mmap");
        return 1;
    }
    long after_map = minor_faults();

    // Touch one byte in each base-page interval. The volatile pointer keeps
    // the compiler from deleting these observable stores.
    for (size_t offset = 0; offset < length; offset += (size_t)page_size) {
        memory[offset] = 1;
    }
    long after_touch = minor_faults();

    printf("minor faults during mmap:  %ld\n", after_map - before_map);
    printf("minor faults during touch: %ld\n", after_touch - after_map);

    if (munmap((void *)memory, length) == -1) {
        perror("munmap");
        return 1;
    }
    return 0;
}
```

```bash
cc -std=c11 -O2 -Wall -Wextra -Wpedantic first_touch.c -o first_touch
./first_touch
/usr/bin/time -v ./first_touch
```

</details>

A **minor fault** can be satisfied without reading the page's contents from storage, while a **major fault** requires storage I/O under the platform's accounting definition. Neither label identifies whether the original program was correct. A protection violation may be fatal without being a major fault; a valid first access may be a minor fault and entirely expected.

### **Comparing x86-64 and RISC-V Translation**

x86-64 and RISC-V expose different encodings and instructions, but both illustrate the same modern pattern: a privileged register selects a radix-tree root and address-space identity; equal-size PTEs fill page-sized tables; VPN fields index successive levels; a leaf provides permissions and a physical page number; and explicit maintenance orders page-table changes with cached translations.

![Conceptual comparison of x86-64 four-level paging and RISC-V Sv39.](assets/x86-riscv-translation.svg){fig-alt="Side-by-side virtual-address fields, root registers, representative PTE state, and invalidation operations for x86-64 four-level paging and RISC-V Sv39." width="98%"}

*Figure: original explanatory diagram based on the [Intel 64 and IA-32 manuals](https://www.intel.com/content/www/us/en/support/articles/000006715/processors.html), Linux [x86-64 memory documentation](https://docs.kernel.org/arch/x86/x86_64/index.html), and the ratified [RISC-V supervisor specification](https://docs.riscv.org/reference/isa/priv/supervisor.html).*

| Property | x86-64, common 4-level mode | RISC-V Sv39 |
|---|---|---|
| Base page | 4 KiB | 4 KiB |
| Translated virtual fields | four 9-bit indexes + 12-bit offset | three 9-bit VPN fields + 12-bit offset |
| Canonical-address rule | upper bits replicate the top translated bit for the selected mode | bits 63:39 equal bit 38 |
| Root state | CR3 supplies paging root; PCID may tag translations | `satp` supplies mode, ASID, and root PPN |
| Leaf permissions | present, read/write, user/supervisor, execute-disable, and other attributes | V, R, W, X, U, G, A, D, plus defined extensions |
| Larger mappings | upper-level leaves for large pages | leaf at any walk level for aligned superpages |
| Maintenance examples | `INVLPG`, `INVPCID`, and CR3-related effects | `SFENCE.VMA` with address/ASID selection |
| Deeper modes | 5-level paging extends translated virtual width | Sv48 and Sv57 add levels |

These maintenance instructions act on the executing logical processor or hart; they do not magically erase stale entries from every other core. The RISC-V specification is especially explicit that `SFENCE.VMA` orders only the local hart's implicit translation references. A multiprocessor OS must first make the page-table update visible as required, notify relevant remote harts, execute appropriate local fences or invalidations there, and collect acknowledgements when correctness requires them. This is the architecture-specific realization of the shootdown protocol shown earlier.

For a 4 KiB mapping, both examples use 512 eight-byte entries in one 4 KiB table page. That numerical similarity should not hide semantic differences. Reserved bits, permission combinations, accessed/dirty behavior, global mappings, privilege overrides, and ordering requirements must be read from the relevant architecture specification.

RISC-V Sv39 makes the teaching arithmetic especially visible: a 39-bit virtual address has `VPN[2]`, `VPN[1]`, `VPN[0]`, and a 12-bit offset. Each VPN field selects one of 512 entries. A non-leaf entry points to the next table; a leaf at level 0 maps 4 KiB, while aligned upper-level leaves map superpages. `satp` also selects the translation mode and address-space identifier.

x86-64 commonly uses four levels for 48-bit canonical addresses, while optional five-level paging extends the translated width. CR3 identifies the paging root and can carry a PCID when enabled. Hardware defines detailed page-fault error information and invalidation effects. Linux then builds architecture-independent abstractions above these mechanisms, including its five-name software hierarchy and folded levels.

The correct lesson is not to memorize one bit diagram as universal. Learn the invariant algorithm, then consult the current manual for exact legal entries and maintenance rules.

### **Building a Minimal Page-Table Walker**

A real user process cannot read arbitrary physical page tables or replace the MMU. A kernel, hypervisor, debugger, or simulator can implement a walker because it has a trusted way to read paging-structure memory. The minimal algorithm needs five explicit inputs:

- virtual address and requested access type;
- current privilege mode;
- selected translation mode and root PPN;
- a trusted physical-memory read operation;
- architecture-specific validation and permission rules.

The following simulation implements the central Sv39 walk for a small byte array used as physical memory. It supports 4 KiB leaves and aligned superpage leaves, checks canonical addresses and basic `V/R/W/X/U/A/D` rules, and preserves the page offset. It deliberately omits extensions such as MXR, SUM, PBMT, NAPOT, PMP/PMA checks, and hardware-managed A/D updates, so it is an educational walker rather than a replacement for the specification.

<details>
<summary><strong>C: a minimal simulated RISC-V Sv39 page-table walker</strong></summary>

```c
#include <inttypes.h>
#include <stdbool.h>
#include <stdint.h>
#include <stdio.h>
#include <string.h>

enum {
    PAGE_SHIFT = 12,
    PAGE_SIZE = 1U << PAGE_SHIFT,
    PT_ENTRIES = 512,
    RAM_PAGES = 16
};

enum {
    PTE_V = 1U << 0,
    PTE_R = 1U << 1,
    PTE_W = 1U << 2,
    PTE_X = 1U << 3,
    PTE_U = 1U << 4,
    PTE_A = 1U << 6,
    PTE_D = 1U << 7
};

typedef enum {
    ACCESS_READ,
    ACCESS_WRITE,
    ACCESS_EXECUTE
} access_t;

typedef enum {
    WALK_OK,
    WALK_NONCANONICAL,
    WALK_INVALID,
    WALK_PERMISSION,
    WALK_AD_MISSING,
    WALK_MISALIGNED_SUPERPAGE,
    WALK_OUT_OF_RAM
} walk_status_t;

static unsigned char ram[RAM_PAGES * PAGE_SIZE];

static bool read_pte(uint64_t table_ppn, uint16_t index, uint64_t *pte) {
    if (index >= PT_ENTRIES) {
        return false;
    }
    uint64_t address = (table_ppn << PAGE_SHIFT) +
                       (uint64_t)index * sizeof *pte;
    if (address > sizeof ram - sizeof *pte) {
        return false;
    }
    memcpy(pte, &ram[address], sizeof *pte);
    return true;
}

static bool write_pte(uint64_t table_ppn, uint16_t index, uint64_t pte) {
    if (index >= PT_ENTRIES) {
        return false;
    }
    uint64_t address = (table_ppn << PAGE_SHIFT) +
                       (uint64_t)index * sizeof pte;
    if (address > sizeof ram - sizeof pte) {
        return false;
    }
    memcpy(&ram[address], &pte, sizeof pte);
    return true;
}

static bool is_canonical_sv39(uint64_t virtual_address) {
    uint64_t upper = virtual_address >> 39;
    uint64_t expected = ((virtual_address >> 38) & 1U)
                            ? ((UINT64_C(1) << 25) - 1U)
                            : 0U;
    return upper == expected;
}

static walk_status_t walk_sv39(uint64_t root_ppn,
                               uint64_t virtual_address,
                               access_t access,
                               bool user_mode,
                               uint64_t *physical_address) {
    if (!is_canonical_sv39(virtual_address)) {
        return WALK_NONCANONICAL;
    }

    uint16_t vpn[3] = {
        (uint16_t)((virtual_address >> 12) & 0x1ffU),
        (uint16_t)((virtual_address >> 21) & 0x1ffU),
        (uint16_t)((virtual_address >> 30) & 0x1ffU)
    };
    uint64_t table_ppn = root_ppn;

    for (int level = 2; level >= 0; --level) {
        uint64_t pte;
        if (!read_pte(table_ppn, vpn[level], &pte)) {
            return WALK_OUT_OF_RAM;
        }

        bool valid = (pte & PTE_V) != 0;
        bool readable = (pte & PTE_R) != 0;
        bool writable = (pte & PTE_W) != 0;
        bool executable = (pte & PTE_X) != 0;

        // Sv39 reserves W=1, R=0 and rejects an entry with V=0.
        if (!valid || (!readable && writable)) {
            return WALK_INVALID;
        }

        if (readable || executable) {
            // A leaf mapping has been reached.
            if (user_mode && (pte & PTE_U) == 0) {
                return WALK_PERMISSION;
            }
            if ((access == ACCESS_READ && !readable) ||
                (access == ACCESS_WRITE && !writable) ||
                (access == ACCESS_EXECUTE && !executable)) {
                return WALK_PERMISSION;
            }
            if ((pte & PTE_A) == 0 ||
                (access == ACCESS_WRITE && (pte & PTE_D) == 0)) {
                return WALK_AD_MISSING;
            }

            uint64_t ppn = pte >> 10;
            uint64_t low_ppn_mask = (level == 0)
                                        ? 0U
                                        : ((UINT64_C(1) << (9 * level)) - 1U);

            // Upper-level leaves require physical superpage alignment.
            if ((ppn & low_ppn_mask) != 0) {
                return WALK_MISALIGNED_SUPERPAGE;
            }

            // For a superpage, lower VPN fields become lower PPN fields.
            ppn = (ppn & ~low_ppn_mask) |
                  ((virtual_address >> PAGE_SHIFT) & low_ppn_mask);
            *physical_address = (ppn << PAGE_SHIFT) |
                                (virtual_address & (PAGE_SIZE - 1U));
            return WALK_OK;
        }

        // A non-leaf entry points to the next page-table page.
        if (level == 0) {
            return WALK_INVALID;
        }
        table_ppn = pte >> 10;
    }

    return WALK_INVALID;
}

int main(void) {
    const uint64_t root_ppn = 1;
    const uint64_t level1_ppn = 2;
    const uint64_t level0_ppn = 3;
    const uint64_t data_ppn = 8;
    const uint64_t virtual_address = UINT64_C(0x12345678);

    uint16_t vpn0 = (uint16_t)((virtual_address >> 12) & 0x1ffU);
    uint16_t vpn1 = (uint16_t)((virtual_address >> 21) & 0x1ffU);
    uint16_t vpn2 = (uint16_t)((virtual_address >> 30) & 0x1ffU);

    // Build root -> level 1 -> level 0 -> data-frame mappings.
    if (!write_pte(root_ppn, vpn2, (level1_ppn << 10) | PTE_V) ||
        !write_pte(level1_ppn, vpn1, (level0_ppn << 10) | PTE_V) ||
        !write_pte(level0_ppn, vpn0,
                   (data_ppn << 10) |
                       PTE_V | PTE_R | PTE_W | PTE_U | PTE_A | PTE_D)) {
        fputs("failed to construct simulated page tables\n", stderr);
        return 1;
    }

    uint64_t physical_address;
    walk_status_t status = walk_sv39(root_ppn, virtual_address,
                                     ACCESS_WRITE, true,
                                     &physical_address);
    if (status != WALK_OK) {
        fprintf(stderr, "walk failed with status %d\n", status);
        return 1;
    }

    printf("VA 0x%08" PRIx64 " -> PA 0x%05" PRIx64 "\n",
           virtual_address, physical_address);
    return 0;
}
```

```bash
cc -std=c11 -O2 -Wall -Wextra -Wpedantic sv39_walk.c -o sv39_walk
./sv39_walk
# VA 0x12345678 -> PA 0x08678
```

</details>

The walk's structure is more important than the example numbers. `VPN[2]` indexes the root, each non-leaf PTE supplies the next physical table page, and the leaf supplies permissions plus a PPN. The final line combines data frame 8 with virtual offset `0x678`, producing physical address `0x8678`.

A production implementation must handle all specification rules, concurrent page-table modification, memory ordering, TLB maintenance, physical-memory access failures, huge mappings, architecture extensions, and attacks involving malformed state. The simulation intentionally makes those omitted obligations visible rather than hiding them behind a one-line array lookup.

### **Comparison and Summary**

The mechanisms in this chapter solve related but different costs:

| Mechanism | Primary purpose | Fast-path strength | Main cost or risk |
|---|---|---|---|
| Base and bounds | relocate and protect one contiguous region | one comparison and addition | inflexible growth and contiguous placement |
| Segmentation | independently place and protect logical regions | small descriptor lookup | external fragmentation and variable-size allocation |
| Paging | map fixed-size virtual pages to arbitrary frames | uniform allocation and fine-grained permissions | page-table memory and translation latency |
| Multilevel page table | avoid storing entries for large virtual holes | regular hardware walk | dependent memory reads on a miss |
| Huge page | increase TLB reach and reduce PTE count | one entry covers much more memory | coarse allocation, protection, copying, and reclaim |
| TLB | cache recent translations and permissions | avoids page-table reads on hits | limited reach, replacement misses, stale-entry maintenance |
| ASID / PCID | distinguish address spaces in the TLB | avoids routine full flushes | finite tag reuse and generation management |
| TLB shootdown | remove stale translations across CPUs | restores permission and lifetime correctness | interprocessor latency and refill collateral |
| Page fault | transfer exceptional translation decisions to the kernel | enables sparse and lazy policies | high handling cost; may reveal a bug or require I/O |

An end-to-end memory reference can now be reasoned about in order:

1. The program and ISA form a virtual address.
2. Region metadata says whether the process logically owns the interval.
3. A process-specific root and optional ASID/PCID select the translation namespace.
4. The TLB attempts to supply a cached leaf translation and permissions.
5. On a miss, the MMU walks multilevel page tables using VPN fields.
6. A valid leaf maps a PFN and preserves the page offset.
7. Permission and privilege checks authorize the operation.
8. Invalid or denied state raises a page fault for kernel classification.
9. Page-table changes require architecture-defined ordering and TLB invalidation.
10. Only then does the resulting physical address proceed through caches and memory.

Several common misconceptions are now easier to reject:

- a virtual address is not a physical address with a constant offset;
- a TLB miss is not automatically a page fault;
- a page fault does not automatically mean disk I/O;
- `/proc/PID/maps` does not prove every page is resident;
- an ASID avoids many flushes but does not remove shootdowns after mapping changes;
- a larger page is not automatically faster for every workload;
- changing a PTE in memory is not sufficient until stale cached translations are handled.

This chapter established **where an address may translate and how that decision is accelerated**. The next chapter starts at the repairable-fault branch and asks a different policy question: when a legal page lacks a resident frame, how should the OS allocate, load, replace, reclaim, or share physical memory?
